In [2]:
import numpy as np 
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import os 

In [ ]:
# Config
DATASET = "Zebrafish-2024"
DATASET_CONFIG = {
    "Mouse-2023": {
        "design": "~ treatment",
        "levels": {
            "treatment": ["control", "CFA", "CFB"]
        },
        "contrasts": [
            ["treatment", "CFA", "control"],
            ["treatment", "CFB", "control"]
        ],
        "shrink_map": {
            "CFA_vs_control": "treatment[T.CFA]",
            "CFB_vs_control": "treatment[T.CFB]"
        }
    },
    "Zebrafish-2024": {
        "design": "~ batch + drug",
        "levels": {
            "drug": ["BMAA", "CNR401", "Edaravone", "CFA"],
            "batch": ["batch_1", "batch_2"]
        },
        "contrasts": [
            ["drug", "CNR401", "BMAA"],
            ["drug", "CFA", "BMAA"],
            ["drug", "Edaravone", "BMAA"]
        ],
        "shrink_map": {
            "CNR401_vs_BMAA": "drug[T.CNR401]",
            "CFA_vs_BMAA": "drug[T.CFA]",
            "Edaravone_vs_BMAA": "drug[T.Edaravone]"
        }
    }
}

PROCESSED_DIR = os.path.join(DATASET, "processed-data")
OUT_DIR = os.path.join(DATASET, "DEA-results")

N_CPUS = 4            # parallel dispersion estimation
ALPHA = 0.05          # Benjamini-Hochberg FDR cutoff
MIN_TOTAL_COUNT = 10  # drop genes with fewer total reads

# A means-reference model is used so the first level is the intercept/baseline
DESIGN = DATASET_CONFIG[DATASET]["design"]
LEVELS_MAP = DATASET_CONFIG[DATASET]["levels"]
COMPARISONS = DATASET_CONFIG[DATASET]["contrasts"]
SHRINK_MAP = DATASET_CONFIG[DATASET]["shrink_map"]

In [4]:
# Read in data 
counts_file = os.path.join(PROCESSED_DIR, "filtered_counts.csv")
metadata_file = os.path.join(PROCESSED_DIR, "filtered_metadata.csv")
# Note that dataframes must be indexed by sample
metadata = pd.read_csv(metadata_file, index_col='sample')
counts = pd.read_csv(counts_file, index_col='sample')

if DATASET == "Zebrafish-2024":
  # Drop healthy controls (also removes all of batch 3)
  metadata = metadata[metadata["drug"] != "none"]
  counts = counts.loc[metadata.index]
  # Drop date_collected column since batch collects these effects better
  metadata = metadata.drop("date_collected", axis=1)

# Drop genes with <10 counts
counts = counts.loc[:, counts.sum(axis=0) >= MIN_TOTAL_COUNT]


In [ ]:
metadata

In [ ]:
counts.shape

In [ ]:
# Set reference levels 
for column, levels in LEVELS_MAP.items():
    if column in metadata.columns:
        metadata[column] = pd.Categorical(
            metadata[column], 
            categories=levels, 
            ordered=True
        )

# Initialize DeseqDataSet
# A single GLM is used for statistical power; better dispersion estimations 
dds = DeseqDataSet(
    counts=counts, 
    metadata=metadata, 
    design = DESIGN,
    fit_type = "parametric",
    size_factors_fit_type= "ratio",
    refit_cooks=True, 
    n_cpus=N_CPUS
)
dds.deseq2()

In [ ]:
# Access and display the full design matrix
design_matrix = dds.obsm["design_matrix"]
design_matrix

In [ ]:
# View a list of the coefficient names
print(dds.obsm["design_matrix"].columns)


Index(['Intercept', 'treatment[T.CFA]', 'treatment[T.CFB]'], dtype='object')


'\n# Zebrafish\nshrink_map = {\n  "CNR401_vs_BMAA": "drug[T.CNR401]",\n  "CFA_vs_BMAA": "drug[T.CFA]",\n  "Edaravone_vs_BMAA": "drug[T.Edaravone]"\n}\n'

In [ ]:
results = {}
for contrast in COMPARISONS:
    res_label = f"{contrast[1]}_vs_{contrast[2]}"
    stat_res = DeseqStats(dds, contrast=contrast, alpha=ALPHA, 
                          cooks_filter=True, independent_filter=True,
                          n_cpus=N_CPUS)
    stat_res.summary()
    raw_res = stat_res.results_df.copy()
   
    # LFC shrinkage
    coeff_name = SHRINK_MAP.get(res_label)
    stat_res.lfc_shrink(coeff=coeff_name, adapt=True)  
    shrunk_res = stat_res.results_df.copy()
    shrunk_res = shrunk_res.rename(columns={
        "log2FoldChange": "LFC_shrunk",
         "lfcSE": "lfcSE_shrunk"})
    shrunk_res["LFC_raw"] = raw_res["log2FoldChange"]
    shrunk_res["lfcSE_raw"] = raw_res["lfcSE"]
    shrunk_res.index.name = "gene_id"
    # Save results
    results[res_label] = shrunk_res
  

In [9]:
for res_label, df in results.items():
    filename = f"DEG_{res_label}.csv"
    filepath = os.path.join(OUT_DIR, filename)
    df.to_csv(filepath)

summaries = []
for res_label, df in results.items():
    # Only count genes that passed the independent filtering (non-NaN padj)
    tested = df.dropna(subset=["padj"])
    sig = tested[tested["padj"] <= ALPHA]
    
    summaries.append({
        "comparison": res_label,
        "n_features": len(df),
        "n_sig": len(sig),
        "n_up": (sig["LFC_shrunk"] > 0).sum(),
        "n_down": (sig["LFC_shrunk"] < 0).sum()
    })

summary_df = pd.DataFrame(summaries).sort_values("n_sig", ascending=False)
summary_df.to_csv(f"{OUT_DIR}/summary_all_comparisons.csv", index=False)